In [ ]:
import os

if not os.path.exists("train-v1.1.json"):
    !wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json
    !wget https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json

print("Files downloaded successfully!")

In [ ]:
SQUAD_PATH = "/content/train-v1.1.json"
DEV_PATH = "/content/dev-v1.1.json"

In [ ]:
import os

print(os.path.exists(SQUAD_PATH))
print(os.path.exists(DEV_PATH))

In [ ]:
import tensorflow as tf
import numpy as np
import json
import re
import random
import collections
from tqdm.auto import tqdm


MAX_SEQ_LEN = 128

D_MODEL = 256
NUM_HEADS = 4
NUM_LAYERS = 4
FFN_DIM = 1024

DROPOUT_RATE = 0.1

VOCAB_SIZE = 30000

BATCH_SIZE = 16

EPOCHS = 3

LEARNING_RATE = 3e-4

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("\nConfiguration:")
print("MAX_SEQ_LEN :", MAX_SEQ_LEN)
print("VOCAB_SIZE  :", VOCAB_SIZE)
print("BATCH_SIZE  :", BATCH_SIZE)



with open(SQUAD_PATH, "r") as f:
    squad = json.load(f)

print("\nSQuAD Loaded Successfully")
print("Articles:", len(squad["data"]))

# ==========================================================
# EXTRACT QA EXAMPLES
# ==========================================================

examples = []

for article in squad["data"]:

    for paragraph in article["paragraphs"]:

        context = paragraph["context"]

        for qa in paragraph["qas"]:

            question = qa["question"]

            answer = qa["answers"][0]

            examples.append(
                {
                    "context": context,
                    "question": question,
                    "answer": answer["text"],
                    "answer_start": answer["answer_start"]
                }
            )

print("Total QA Examples:", len(examples))


def basic_tokenizer(text):

    text = text.lower()

    text = re.sub(
        r"([.,!?;:()\"'])",
        r" \1 ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip().split()

sample = "Hello, World! This is MiniBERT."

print("\nTokenizer Test:")

print("Input:")
print(sample)

print("\nOutput:")
print(
    basic_tokenizer(sample)
)

print("\nFirst Example:")

print(
    "Question:",
    examples[0]["question"]
)

print(
    "Answer:",
    examples[0]["answer"]
)

print(
    "Context Preview:",
    examples[0]["context"][:200]
)

In [ ]:
counter = collections.Counter()

print("Building vocabulary...")

for ex in tqdm(examples):

    counter.update(
        basic_tokenizer(
            ex["context"]
        )
    )

    counter.update(
        basic_tokenizer(
            ex["question"]
        )
    )

print("Unique Tokens:", len(counter))

vocab = {
    "[PAD]": 0,
    "[UNK]": 1,
    "[CLS]": 2,
    "[SEP]": 3
}

for token, _ in counter.most_common(
    VOCAB_SIZE - len(vocab)
):

    vocab[token] = len(vocab)


id2word = {
    idx: word
    for word, idx in vocab.items()
}

print("Final Vocabulary Size:", len(vocab))

def encode_tokens(tokens):

    return [
        vocab.get(
            token,
            vocab["[UNK]"]
        )
        for token in tokens
    ]

def find_answer_span(
    context_tokens,
    answer_tokens
):

    answer_len = len(answer_tokens)

    for i in range(
        len(context_tokens)
        - answer_len + 1
    ):

        if (
            context_tokens[i:i+answer_len]
            == answer_tokens
        ):

            return i, i + answer_len - 1

    return None, None

def process_example(example):

    question_tokens = basic_tokenizer(
        example["question"]
    )

    context_tokens = basic_tokenizer(
        example["context"]
    )

    answer_tokens = basic_tokenizer(
        example["answer"]
    )

    c_start, c_end = find_answer_span(
        context_tokens,
        answer_tokens
    )

    if c_start is None:
        return None


    tokens = (
        ["[CLS]"]
        + question_tokens
        + ["[SEP]"]
        + context_tokens
        + ["[SEP]"]
    )


    token_type_ids = (
        [0] * (
            len(question_tokens) + 2
        )
        +
        [1] * (
            len(context_tokens) + 1
        )
    )


    context_offset = (
        len(question_tokens)
        + 2
    )

    start_pos = (
        context_offset
        + c_start
    )

    end_pos = (
        context_offset
        + c_end
    )


    # Remove too-long examples
    if (
        len(tokens) > MAX_SEQ_LEN
        or end_pos >= MAX_SEQ_LEN
    ):

        return None


    input_ids = encode_tokens(
        tokens
    )


    attention_mask = (
        [1]
        * len(input_ids)
    )


    pad_len = (
        MAX_SEQ_LEN
        - len(input_ids)
    )


    input_ids += (
        [vocab["[PAD]"]]
        * pad_len
    )

    token_type_ids += (
        [0]
        * pad_len
    )

    attention_mask += (
        [0]
        * pad_len
    )


    return (
        input_ids,
        token_type_ids,
        attention_mask,
        start_pos,
        end_pos
    )

processed = []

print("\nProcessing SQuAD examples...")

for ex in tqdm(examples):

    item = process_example(ex)

    if item is not None:
        processed.append(item)


print("\nUsable Examples:", len(processed))

print(
    "Filtered Examples:",
    len(examples) - len(processed)
)

sample = processed[0]

print("\nSample Shapes:")

print(
    "Input IDs:",
    len(sample[0])
)

print(
    "Token Type IDs:",
    len(sample[1])
)

print(
    "Attention Mask:",
    len(sample[2])
)

print(
    "Start Position:",
    sample[3]
)

print(
    "End Position:",
    sample[4]
)

decoded = [
    id2word[idx]
    for idx in sample[0]
    if idx != vocab["[PAD]"]
]

print("\nDecoded Input Preview:\n")

print(
    " ".join(decoded[:80])
)

In [ ]:
print("Splitting dataset...")

random.shuffle(processed)

split_idx = int(
    0.9 * len(processed)
)

train_data = processed[:split_idx]

val_data = processed[split_idx:]

print("\nDataset Statistics")
print("-------------------")
print("Total Examples :", len(processed))
print("Train Examples :", len(train_data))
print("Validation Examples :", len(val_data))

def make_dataset(data, training=True):

    input_ids = np.array(
        [x[0] for x in data],
        dtype=np.int32
    )

    token_type_ids = np.array(
        [x[1] for x in data],
        dtype=np.int32
    )

    attention_mask = np.array(
        [x[2] for x in data],
        dtype=np.int32
    )

    start_positions = np.array(
        [x[3] for x in data],
        dtype=np.int32
    )

    end_positions = np.array(
        [x[4] for x in data],
        dtype=np.int32
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            (
                input_ids,
                token_type_ids,
                attention_mask
            ),
            (
                start_positions,
                end_positions
            )
        )
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=10000,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=False
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

print("\nCreating tf.data pipelines...")

train_dataset = make_dataset(
    train_data,
    training=True
)

val_dataset = make_dataset(
    val_data,
    training=False
)

print("Done!")

train_steps = int(
    np.ceil(
        len(train_data) / BATCH_SIZE
    )
)

val_steps = int(
    np.ceil(
        len(val_data) / BATCH_SIZE
    )
)

print("\nSteps Information")
print("------------------")
print("Train Steps :", train_steps)
print("Validation Steps :", val_steps)

print("\nRunning Sanity Check...")

for features, targets in train_dataset.take(1):

    input_ids, token_type_ids, attention_mask = features

    y_start, y_end = targets

    print("\nFeature Shapes")
    print("----------------")
    print("Input IDs       :", input_ids.shape)
    print("Token Type IDs  :", token_type_ids.shape)
    print("Attention Mask  :", attention_mask.shape)

    print("\nTarget Shapes")
    print("----------------")
    print("Start Positions :", y_start.shape)
    print("End Positions   :", y_end.shape)

    print("\nFirst Example")
    print("----------------")
    print("Start Position :", y_start[0].numpy())
    print("End Position   :", y_end[0].numpy())

    break



print("\nPart 1 Completed Successfully!")
print("--------------------------------")
print("Vocabulary Size :", len(vocab))
print("Sequence Length :", MAX_SEQ_LEN)
print("Batch Size      :", BATCH_SIZE)

print("\nMiniBERT QA Data Pipeline Ready ")

In [ ]:

# Embeddings + Multi-Head Self Attention

class BertEmbeddings(tf.keras.layers.Layer):

    def __init__(
        self,
        vocab_size,
        d_model,
        max_seq_len,
        dropout_rate=0.1
    ):

        super().__init__()

        self.token_embeddings = tf.keras.layers.Embedding(
            vocab_size,
            d_model
        )

        self.position_embeddings = tf.keras.layers.Embedding(
            max_seq_len,
            d_model
        )

        self.segment_embeddings = tf.keras.layers.Embedding(
            2,
            d_model
        )

        self.layer_norm = tf.keras.layers.LayerNormalization(
            epsilon=1e-12
        )

        self.dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

    def call(
        self,
        input_ids,
        token_type_ids,
        training=False
    ):

        seq_len = tf.shape(input_ids)[1]

        position_ids = tf.range(
            seq_len,
            dtype=tf.int32
        )

        position_ids = tf.expand_dims(
            position_ids,
            axis=0
        )

        token_emb = self.token_embeddings(
            input_ids
        )

        position_emb = self.position_embeddings(
            position_ids
        )

        segment_emb = self.segment_embeddings(
            token_type_ids
        )

        embeddings = (
            token_emb
            + position_emb
            + segment_emb
        )

        embeddings = self.layer_norm(
            embeddings
        )

        embeddings = self.dropout(
            embeddings,
            training=training
        )

        return embeddings



# MULTI-HEAD SELF ATTENTION

class MultiHeadSelfAttention(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        d_model,
        num_heads,
        dropout_rate=0.1
    ):

        super().__init__()

        assert (
            d_model % num_heads == 0
        ), "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads

        self.head_dim = (
            d_model // num_heads
        )

        self.q_dense = tf.keras.layers.Dense(
            d_model
        )

        self.k_dense = tf.keras.layers.Dense(
            d_model
        )

        self.v_dense = tf.keras.layers.Dense(
            d_model
        )

        self.output_dense = tf.keras.layers.Dense(
            d_model
        )

        self.attn_dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

    def split_heads(
        self,
        x,
        batch_size
    ):

        x = tf.reshape(
            x,
            (
                batch_size,
                -1,
                self.num_heads,
                self.head_dim
            )
        )

        return tf.transpose(
            x,
            perm=[0, 2, 1, 3]
        )

    def call(
        self,
        hidden_states,
        attention_mask,
        training=False
    ):

        batch_size = tf.shape(
            hidden_states
        )[0]

        Q = self.q_dense(
            hidden_states
        )

        K = self.k_dense(
            hidden_states
        )

        V = self.v_dense(
            hidden_states
        )

        Q = self.split_heads(
            Q,
            batch_size
        )

        K = self.split_heads(
            K,
            batch_size
        )

        V = self.split_heads(
            V,
            batch_size
        )


        scores = tf.matmul(
            Q,
            K,
            transpose_b=True
        )

        scores = tf.cast(scores, tf.float32) / tf.math.sqrt(
            tf.cast(
                self.head_dim,
                tf.float32
            )
        )


        # Proper Attention Mask

        mask = tf.cast(
            attention_mask,
            tf.float32
        )

        mask = mask[
            :,
            tf.newaxis,
            tf.newaxis,
            :
        ]

        scores += (
            1.0 - mask
        ) * (-1e9)

        attention_weights = tf.nn.softmax(
            scores,
            axis=-1
        )

        attention_weights = self.attn_dropout(
            attention_weights,
            training=training
        )

        context = tf.matmul(
            attention_weights,
            V
        )

        context = tf.transpose(
            context,
            perm=[0, 2, 1, 3]
        )

        context = tf.reshape(
            context,
            (
                batch_size,
                -1,
                self.d_model
            )
        )

        output = self.output_dense(
            context
        )

        return (
            output,
            attention_weights
        )


In [ ]:

# FFN + Encoder Layer + MiniBERT + QA Head

# FEED FORWARD NETWORK

class FeedForwardNetwork(tf.keras.layers.Layer):

    def __init__(
        self,
        d_model,
        ffn_dim,
        dropout_rate=0.1
    ):

        super().__init__()

        self.dense1 = tf.keras.layers.Dense(
            ffn_dim,
            activation=tf.nn.gelu
        )

        self.dense2 = tf.keras.layers.Dense(
            d_model
        )

        self.dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

    def call(
        self,
        x,
        training=False
    ):

        x = self.dense1(x)

        x = self.dense2(x)

        x = self.dropout(
            x,
            training=training
        )

        return x

# TRANSFORMER ENCODER LAYER

class EncoderLayer(tf.keras.layers.Layer):

    def __init__(
        self,
        d_model,
        num_heads,
        ffn_dim,
        dropout_rate=0.1
    ):

        super().__init__()

        self.attention = MultiHeadSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout_rate=dropout_rate
        )

        self.ffn = FeedForwardNetwork(
            d_model=d_model,
            ffn_dim=ffn_dim,
            dropout_rate=dropout_rate
        )

        self.norm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-12
        )

        self.norm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-12
        )

    def call(
        self,
        hidden_states,
        attention_mask,
        training=False
    ):

        # Self Attention

        attention_output, _ = self.attention(
            hidden_states,
            attention_mask,
            training=training
        )

        hidden_states = self.norm1(
            hidden_states + attention_output
        )

        # Feed Forward

        ffn_output = self.ffn(
            hidden_states,
            training=training
        )

        hidden_states = self.norm2(
            hidden_states + ffn_output
        )

        return hidden_states



# MINIBERT ENCODER

class MiniBERTEncoder(tf.keras.layers.Layer):

    def __init__(
        self,
        vocab_size,
        max_seq_len,
        d_model,
        num_heads,
        num_layers,
        ffn_dim,
        dropout_rate=0.1
    ):

        super().__init__()

        self.embeddings = BertEmbeddings(
            vocab_size=vocab_size,
            d_model=d_model,
            max_seq_len=max_seq_len,
            dropout_rate=dropout_rate
        )

        self.encoder_layers = [

            EncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                dropout_rate=dropout_rate
            )

            for _ in range(num_layers)
        ]

    def call(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        training=False
    ):

        hidden_states = self.embeddings(
            input_ids,
            token_type_ids,
            training=training
        )

        for layer in self.encoder_layers:

            hidden_states = layer(
                hidden_states,
                attention_mask,
                training=training
            )

        return hidden_states

# MINIBERT FOR QUESTION ANSWERING

class MiniBERTQA(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_seq_len,
        d_model,
        num_heads,
        num_layers,
        ffn_dim,
        dropout_rate=0.1
    ):

        super().__init__()

        self.encoder = MiniBERTEncoder(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            ffn_dim=ffn_dim,
            dropout_rate=dropout_rate
        )

        # QA Head
        self.qa_outputs = tf.keras.layers.Dense(
            2
        )

    def call(
        self,
        input_ids,
        token_type_ids,
        attention_mask,
        training=False
    ):

        sequence_output = self.encoder(
            input_ids,
            token_type_ids,
            attention_mask,
            training=training
        )

        logits = self.qa_outputs(
            sequence_output
        )

        # logits:
        # (batch, seq_len, 2)

        start_logits = logits[:, :, 0]

        end_logits = logits[:, :, 1]

        return (
            start_logits,
            end_logits
        )

# CREATE MODEL

model = MiniBERTQA(
    vocab_size=len(vocab),
    max_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ffn_dim=FFN_DIM,
    dropout_rate=DROPOUT_RATE
)

# BUILD MODEL

dummy_input_ids = tf.zeros(
    (1, MAX_SEQ_LEN),
    dtype=tf.int32
)

dummy_token_type_ids = tf.zeros(
    (1, MAX_SEQ_LEN),
    dtype=tf.int32
)

dummy_attention_mask = tf.ones(
    (1, MAX_SEQ_LEN),
    dtype=tf.int32
)

_ = model(
    dummy_input_ids,
    dummy_token_type_ids,
    dummy_attention_mask,
    training=False
)

model.summary()


In [ ]:

# Optimizer + Scheduler + Loss + Metrics

import math
import tensorflow.keras.mixed_precision as mixed_precision


TOTAL_TRAIN_STEPS = train_steps * EPOCHS

WARMUP_STEPS = int(
    0.1 * TOTAL_TRAIN_STEPS
)

print("Training Configuration")
print("-----------------------")
print("Total Train Steps :", TOTAL_TRAIN_STEPS)
print("Warmup Steps      :", WARMUP_STEPS)

class WarmupLinearDecay(
    tf.keras.optimizers.schedules.LearningRateSchedule
):

    def __init__(
        self,
        initial_lr,
        total_steps,
        warmup_steps
    ):

        super().__init__()

        self.initial_lr = initial_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps

    def __call__(
        self,
        step
    ):

        step = tf.cast(
            step,
            tf.float32
        )

        warmup_lr = (
            self.initial_lr
            * (
                step
                /
 tf.cast(
                    self.warmup_steps,
                    tf.float32
                )
            )
        )

        decay_lr = (
            self.initial_lr
            *
 (
                1.0
                -
                (
                    (
                        step
                        -
                        self.warmup_steps
                    )
                    /
                    tf.cast(
                        self.total_steps
                        -
                        self.warmup_steps,
                        tf.float32
                    )
                )
            )
        )

        lr = tf.where(
            step < self.warmup_steps,
            warmup_lr,
            decay_lr
        )

        lr = tf.maximum(
            lr,
            0.0
        )

        return lr

lr_schedule = WarmupLinearDecay(
    initial_lr=LEARNING_RATE,
    total_steps=TOTAL_TRAIN_STEPS,
    warmup_steps=WARMUP_STEPS
)

base_optimizer = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-6
)

# Mixed Precision Support
optimizer = mixed_precision.LossScaleOptimizer(
    base_optimizer
)

print("\nOptimizer Created")

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

print("Loss Function Created")

def compute_loss(
    y_start,
    y_end,
    start_logits,
    end_logits
):

    start_loss = loss_fn(
        y_start,
        start_logits
    )

    end_loss = loss_fn(
        y_end,
        end_logits
    )

    total_loss = (
        start_loss
        +
        end_loss
    )

    return tf.reduce_mean(
        total_loss
    )

def compute_accuracy(
    y_true,
    logits
):

    predictions = tf.argmax(
        logits,
        axis=1,
        output_type=tf.int32
    )

    accuracy = tf.reduce_mean(
        tf.cast(
            predictions == y_true,
            tf.float32
        )
    )

    return accuracy

def exact_match_score(
    pred_start,
    pred_end,
    true_start,
    true_end
):

    return int(
        (
            pred_start == true_start
        )
        and
        (
            pred_end == true_end
        )
    )

def f1_score(
    pred_start,
    pred_end,
    true_start,
    true_end
):

    pred_tokens = set(
        range(
            pred_start,
            pred_end + 1
        )
    )

    true_tokens = set(
        range(
            true_start,
            true_end + 1
        )
    )

    common = pred_tokens.intersection(
        true_tokens
    )

    if len(common) == 0:

        return 0.0

    precision = (
        len(common)
        /
        len(pred_tokens)
    )

    recall = (
        len(common)
        /
        len(true_tokens)
    )

    return (
        2
        *
 precision
        *
 recall
        /
        (
            precision
            + recall
        )
    )

def compute_em_f1(
    start_logits,
    end_logits,
    y_start,
    y_end
):

    pred_start = tf.argmax(
        start_logits,
        axis=1
    ).numpy()

    pred_end = tf.argmax(
        end_logits,
        axis=1
    ).numpy()

    true_start = y_start.numpy()

    true_end = y_end.numpy()

    em_total = 0.0
    f1_total = 0.0

    batch_size = len(
        pred_start
    )

    for i in range(
        batch_size
    ):

        em_total += exact_match_score(
            pred_start[i],
            pred_end[i],
            true_start[i],
            true_end[i]
        )

        f1_total += f1_score(
            pred_start[i],
            pred_end[i],
            true_start[i],
            true_end[i]
        )

    em = em_total / batch_size

    f1 = f1_total / batch_size

    return em, f1

In [ ]:

# Training + Validation + Checkpointing

import os

CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_val_f1 = 0.0

@tf.function
def train_step(
    input_ids,
    token_type_ids,
    attention_mask,
    y_start,
    y_end
):

    with tf.GradientTape() as tape:

        start_logits, end_logits = model(
            input_ids,
            token_type_ids,
            attention_mask,
            training=True
        )

        loss = compute_loss(
            y_start,
            y_end,
            start_logits,
            end_logits
        )

        scaled_loss = optimizer.scale_loss(
            loss
        )

    gradients = tape.gradient(
        scaled_loss,
        model.trainable_variables
    )

    # The LossScaleOptimizer automatically unscales gradients during apply_gradients.
    # So, we remove the explicit call to get_unscaled_gradients.
    # gradients = optimizer.get_unscaled_gradients(
    #     scaled_gradients
    # )

    gradients = [
        tf.clip_by_norm(g, 1.0)
        if g is not None else None
        for g in gradients
    ]

    optimizer.apply_gradients(
        zip(
            gradients,
            model.trainable_variables
        )
    )

    start_acc = compute_accuracy(
        y_start,
        start_logits
    )

    end_acc = compute_accuracy(
        y_end,
        end_logits
    )

    return (
        loss,
        start_acc,
        end_acc
    )

@tf.function
def val_step(
    input_ids,
    token_type_ids,
    attention_mask
):

    start_logits, end_logits = model(
        input_ids,
        token_type_ids,
        attention_mask,
        training=False
    )

    return (
        start_logits,
        end_logits
    )

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("=" * 60)

    train_loss = 0.0
    train_start_acc = 0.0
    train_end_acc = 0.0

    train_batches = 0

    for features, targets in train_dataset:

        input_ids, token_type_ids, attention_mask = features
        y_start, y_end = targets

        loss, start_acc, end_acc = train_step(
            input_ids,
            token_type_ids,
            attention_mask,
            y_start,
            y_end
        )

        train_loss += loss.numpy()
        train_start_acc += start_acc.numpy()
        train_end_acc += end_acc.numpy()

        train_batches += 1

    train_loss /= train_batches
    train_start_acc /= train_batches
    train_end_acc /= train_batches

    val_loss = 0.0
    val_start_acc = 0.0
    val_end_acc = 0.0

    val_em = 0.0
    val_f1 = 0.0

    val_batches = 0

    for features, targets in val_dataset:

        input_ids, token_type_ids, attention_mask = features
        y_start, y_end = targets

        start_logits, end_logits = val_step(
            input_ids,
            token_type_ids,
            attention_mask
        )

        loss = compute_loss(
            y_start,
            y_end,
            start_logits,
            end_logits
        )

        start_acc = compute_accuracy(
            y_start,
            start_logits
        )

        end_acc = compute_accuracy(
            y_end,
            end_logits
        )

        em, f1 = compute_em_f1(
            start_logits,
            end_logits,
            y_start,
            y_end
        )

        val_loss += loss.numpy()
        val_start_acc += start_acc.numpy()
        val_end_acc += end_acc.numpy()

        val_em += em
        val_f1 += f1

        val_batches += 1

    val_loss /= val_batches
    val_start_acc /= val_batches
    val_end_acc /= val_batches

    val_em /= val_batches
    val_f1 /= val_batches

    print(
        f"Train Loss      : {train_loss:.4f}"
    )

    print(
        f"Train Start Acc : {train_start_acc:.4f}"
    )

    print(
        f"Train End Acc   : {train_end_acc:.4f}"
    )

    print()

    print(
        f"Val Loss        : {val_loss:.4f}"
    )

    print(
        f"Val Start Acc   : {val_start_acc:.4f}"
    )

    print(
        f"Val End Acc     : {val_end_acc:.4f}"
    )

    print(
        f"Val EM          : {val_em:.4f}"
    )

    print(
        f"Val F1          : {val_f1:.4f}"
    )


    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        save_path = os.path.join(
            CHECKPOINT_DIR,
            "best_model.weights.h5"
        )

        model.save_weights(
            save_path
        )

        print(
            f"\nBest model saved! "
            f"(F1={val_f1:.4f})"
        )


print("\nTraining Complete!")

print(
    f"Best Validation F1: {best_val_f1:.4f}"
)
